# CyberSentryAI — Security Log Analyzer (offline fallback)

**IS2083 Lab 5 | Partial-completion backup for VPN outage**

**Important:** This backup is a *rule-based* security log analyzer, **not** a successful UTSA Qwen Coder connection. It lets you process a genuine Windows security log, generate a report, and prepare a GitHub portfolio while waiting for approved VPN/JupyterHub access. **It does not fulfill the Qwen connection or Qwen-assistance screenshots.**

**Safety:** Never put an API key or raw security log in a public GitHub repository. Do not share keys in screenshots. Verify the output before you publish.

**Screenshot:** capture the notebook after opening it in a Jupyter-compatible environment.


## 1. Purpose and workflow

**Input:** exported Windows Security log as CSV, prepared on your own Windows computer.

**Output:** `BENIGN`, `SUSPICIOUS`, or `MALICIOUS` for each reviewed event, plus short explanation and `assessment.md` with aggregate results.

**How it works:** An explicit set of triage rules maps Windows Event IDs to tentative labels, and a separate rule looks for suspicious command-line patterns. These are **heuristics**, not AI conclusions or proof of an attack. When Qwen becomes available, replace this fallback classifier with your Qwen-backed notebook.


In [ ]:
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
import csv
import re

TOOL_NAME = "CyberSentryAI"
print(f"{TOOL_NAME} ready (offline rule-based mode; Qwen NOT connected)")


## 2. How AI helped design the approach (not Qwen)

During project preparation, a general-purpose AI assistant helped propose separating log parsing, event classification, and reporting, as well as marking all findings as tentative. This **does not substitute** for the assignment's requirement to show the class Qwen Coder model assisting with tool design. Re-run the design prompt in the original UTSA notebook once access is granted.


## 3. Export a Windows log and upload it

In Windows: press **Win + R**, enter `eventvwr.msc`, expand **Windows Logs**, click **Security**; then **Save All Events As...**, choose CSV if offered, and save `security_log.csv`. If Windows won't let you save the log, do not change security settings to force it. Ask your instructor for an approved alternative.

In Google Colab, run the next cell and select the file. The upload exists only in your notebook session. **Do not add the raw CSV to GitHub.** If running outside Colab, place the CSV next to the notebook and set `LOG_FILE` manually.


In [ ]:
LOG_FILE = "security_log.csv"
try:
    from google.colab import files
    print("Select your exported Windows Security CSV. Do not publish it to GitHub.")
    uploaded = files.upload()
    if uploaded:
        LOG_FILE = next(iter(uploaded))
        print("File received for local analysis in this session.")
except ImportError:
    print("Not using Google Colab. Place security_log.csv next to this notebook.")
print("File exists:", Path(LOG_FILE).exists())


## 4. Parse and triage the log

This tool analyzes at most 300 records to keep the run quick. Event ID classification is suggestive; a single event rarely proves malicious activity. Review potentially sensitive output before sharing.


In [ ]:
# Event ID values can be stored under different CSV column names.
EVENT_ID_NAMES = {"event id", "eventid", "id", "event_id"}
EVENT_RULES = {
    4624: ("BENIGN", "Successful logon was recorded; review context if unexpected."),
    4634: ("BENIGN", "A user session logged off."),
    4647: ("BENIGN", "A user initiated logoff."),
    4672: ("SUSPICIOUS", "Special privileges assigned; check account and session context."),
    4625: ("SUSPICIOUS", "A failed logon occurred; repeated failures can indicate guessing."),
    4648: ("SUSPICIOUS", "Explicit credentials were used; verify this was expected."),
    4688: ("BENIGN", "Process creation was recorded; inspect command line for unusual activity."),
    4697: ("SUSPICIOUS", "A service was installed; verify installer and source."),
    4719: ("SUSPICIOUS", "Audit policy changed; investigate whether it was authorized."),
    4720: ("SUSPICIOUS", "A user account was created; verify authorization."),
    4728: ("SUSPICIOUS", "An account was added to a privileged group; verify authorization."),
    4732: ("SUSPICIOUS", "A member was added to a local group; verify authorization."),
    4771: ("SUSPICIOUS", "Kerberos pre-authentication failure recorded."),
    1102: ("MALICIOUS", "Security audit log cleared; high-priority review required (may be authorized)."),
}
SUSPICIOUS_COMMAND = re.compile(r"(?i)(-enc(?:odedcommand)?\b|frombase64string\s*\(|invoke-expression|\biex\s*\()")

def extract_event_id(row):
    for name, value in row.items():
        if str(name).strip().lower() in EVENT_ID_NAMES:
            m = re.search(r"\d+", str(value))
            if m:
                return int(m.group())
    # Event Viewer's CSV often includes Event ID as the third or fourth field.
    return None

@dataclass
class Finding:
    event_id: int | None
    label: str
    reason: str

def classify_row(row):
    event_id = extract_event_id(row)
    label, reason = EVENT_RULES.get(event_id, ("BENIGN", "No high-risk indicator matched; context is still needed."))
    text = " ".join(str(v) for v in row.values())
    if SUSPICIOUS_COMMAND.search(text) and label != "MALICIOUS":
        label, reason = "SUSPICIOUS", "Encoded or command-execution pattern spotted; review the process context."
    return Finding(event_id, label, reason)

MAX_ROWS = 300
if not Path(LOG_FILE).is_file():
    raise FileNotFoundError("Upload your real Windows Security CSV before running this cell.")
with open(LOG_FILE, newline="", encoding="utf-8-sig", errors="replace") as fh:
    reader = csv.DictReader(fh)
    if not reader.fieldnames:
        raise ValueError("No CSV header found. Re-export the log as CSV.")
    findings = [classify_row(row) for _, row in zip(range(MAX_ROWS), reader)]
if not findings:
    raise ValueError("CSV contained no event records.")

print("Entries analyzed:", len(findings))
print("Classification counts:", dict(Counter(item.label for item in findings)))
print("Note: Qwen model was not used in this offline run.")


## 5. Generate a privacy-conscious assessment report

The public report contains **aggregate event IDs, counts, and generic interpretations**. It does not copy event messages, account identifiers, hosts, timestamps, source IPs, or other raw log content.


In [ ]:
counts = Counter(x.label for x in findings)
ids = Counter(x.event_id for x in findings if x.event_id is not None)
flagged = Counter((x.event_id, x.label, x.reason) for x in findings if x.label != "BENIGN")
report_lines = [
    "# CyberSentryAI — Security Assessment Report",
    "",
    f"Generated (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M')}",
    "",
    "## Scope and method",
    f"Reviewed {len(findings)} exported Windows Security event records (maximum {MAX_ROWS}).",
    "Offline RULE-BASED analysis because the UTSA VPN/Qwen connection was unavailable.",
    "No AI model was queried. These labels are preliminary and require human verification.",
    "",
    "## Classification summary",
    f"- BENIGN: {counts.get('BENIGN', 0)}",
    f"- SUSPICIOUS: {counts.get('SUSPICIOUS', 0)}",
    f"- MALICIOUS: {counts.get('MALICIOUS', 0)}",
    "",
    "## Observed event categories (event IDs only)",
]
for event_id, count in ids.most_common(12):
    report_lines.append(f"- Windows Event ID {event_id}: {count} record(s)")
report_lines += ["", "## Findings requiring manual review"]
if flagged:
    for (event_id, label, reason), count in flagged.most_common(12):
        report_lines.append(f"- {label} / Event ID {event_id if event_id is not None else 'unavailable'} ({count}): {reason}")
else:
    report_lines.append("- No flagged patterns matched these basic rules. This does not establish the absence of threats.")
report_lines += [
    "", "## Recommendations",
    "1. Confirm the context of flagged event IDs in the original Event Viewer log.",
    "2. Investigate repeated authentication failures and unexpected account/privilege changes.",
    "3. Correlate with endpoint logs and the relevant system owner before declaring an attack.",
    "4. Re-run the assignment's Qwen model workflow after VPN access is restored.",
    "", "## Limitations and privacy",
    "This output is from deterministic triage rules, not UTSA Qwen Coder. Event IDs alone do not prove malicious activity.",
    "Raw logs were deliberately excluded from this report; do not publish the original CSV.",
]
Path('assessment.md').write_text('\n'.join(report_lines) + '\n', encoding='utf-8')
print('Generated assessment.md')
print('\n'.join(report_lines[:16]))


## 6. GitHub and submission instructions

1. Open GitHub and create **public** repository **CyberSentryAI**.
2. Download `CyberSentryAI_OFFLINE_FALLBACK.ipynb` (File > Download) and `assessment.md` from the Jupyter/Colab file browser.
3. Upload **only those two reviewed files**; **never upload** `security_log.csv`, credentials, screenshots showing credentials, or any notebook with a hardcoded secret. Clear notebook cell outputs before publishing if any contain sensitive details.
4. Screenshot the repository page with both files visible.
5. Add the seven screenshots to your professor's Lab 5 submission template, **but do not claim Qwen connection or Qwen assistance screenshots unless those actions actually succeeded**. Explain the access blocker and request permission to complete missing parts.
6. Export the template to a single PDF; submit that PDF and the public repository URL, as required.

**If Colab is used:** File > Download > Download .ipynb. For the report, click the Files panel and download `assessment.md`.
